In [20]:
import pandas as pd
import pyodbc
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [21]:
nltk.download("vader_lexicon")

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\ab754\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [22]:
sia=SentimentIntensityAnalyzer()

In [23]:
conn=pyodbc.connect(
    'DRIVER={SQL server};'
    'server=DESKTOP-4KB4COA;'
    'DATABASE=PortfolioProject_MarketingAnalytics;'
    'trusted_connection=yes'
)
print("connected successfully")
    

connected successfully


In [24]:
query="select*from dbo.customer_reviews"

In [25]:
query

'select*from dbo.customer_reviews'

In [26]:
pd.read_sql(query,conn)

C:\Users\ab754\AppData\Local\Temp\ipykernel_3792\2097581445.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(query,conn)


,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText
0,1,77,18,2023-12-23,3,"Average experience, nothing special."
1,2,80,19,2024-12-25,5,The quality is top-notch.
2,3,50,13,2025-01-26,4,Five stars for the quick delivery.
3,4,78,15,2025-04-21,3,"Good quality, but could be cheaper."
4,5,64,2,2023-07-16,3,"Average experience, nothing special."
...,...,...,...,...,...,...
1358,1359,28,4,2023-05-25,3,Not worth the money.
1359,1360,58,12,2023-11-13,2,"Average experience, nothing special."
1360,1361,96,15,2023-03-07,5,Customer support was very helpful.
1361,1362,99,2,2025-12-03,1,Product did not meet my expectations.


In [27]:
sia=SentimentIntensityAnalyzer()

In [28]:
text="average experience,nothibg special."

In [29]:
score=sia.polarity_scores(text)

In [30]:
score

{'neg': 0.0, 'neu': 0.426, 'pos': 0.574, 'compound': 0.4019}

In [52]:
def fetch_data_from_sql():
    conn_str =(
        'DRIVER={SQL server};'
        'server=DESKTOP-4KB4COA;'
        'DATABASE=PortfolioProject_MarketingAnalytics;'
        'trusted_connection=yes')
    conn =pyodbc.connect(conn_str)
    query="select *from dbo.customer_reviews"
    df=pd.read_sql(query,conn)
    conn.close()
    return df
        
        
        
        

In [32]:
customer_reviews_df=fetch_data_from_sql()

C:\Users\ab754\AppData\Local\Temp\ipykernel_3792\1603115341.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(query,conn)


In [33]:
def calculate_sentiment(review):
    sentiment=sia.polarity_scores(review)
    return sentiment['compound']

In [34]:
customer_reviews_df["sentiment_score"]= customer_reviews_df["ReviewText"].apply(calculate_sentiment)

In [35]:
customer_reviews_df

,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText,sentiment_score
0,1,77,18,2023-12-23,3,"Average experience, nothing special.",-0.3089
1,2,80,19,2024-12-25,5,The quality is top-notch.,0.0000
2,3,50,13,2025-01-26,4,Five stars for the quick delivery.,0.0000
3,4,78,15,2025-04-21,3,"Good quality, but could be cheaper.",0.2382
4,5,64,2,2023-07-16,3,"Average experience, nothing special.",-0.3089
...,...,...,...,...,...,...,...
1358,1359,28,4,2023-05-25,3,Not worth the money.,-0.1695
1359,1360,58,12,2023-11-13,2,"Average experience, nothing special.",-0.3089
1360,1361,96,15,2023-03-07,5,Customer support was very helpful.,0.6997
1361,1362,99,2,2025-12-03,1,Product did not meet my expectations.,0.0000


In [36]:
# using rating and score to categories the sentiment 

score>0.5  and rating >=4 -->positive
score>0.05 and rating =3 --> mixed positive
score>0.05 and rating <3--> mixed negative

score<0.5  and rating <=2--> negative
score<0.05 and rating =3--> mixed negative
score<0.5 and rating >3--> mixed positive

score other than these and rating >=4 --> positive
score other than these and rating <=2-->  negative

for other cases nutral

SyntaxError: invalid syntax (3987587129.py, line 3)

In [37]:
def sentiment_bucket(score):
    if score >=0.5:
        return '0.5 to 1.0'
    elif 0.0 <=score<0.5:
        return '0.0 to 0.49'
    elif -0.5 <=score <0.0:
        return '-0.49 to -0.5'
    else:
        return '-1.0 to -0.5'
        

In [42]:
def categorize_sentiment(score,rating):
    if score >0.05:
        if rating>=4:
            return "Positive"
        elif rating>=3:
            return "Mixed Positive"
        else:
            return "Negative"
            
    elif score < -0.05:
        if rating<=2:
            return "Negative"
        elif rating==3:
            return "Mixed Negative"
        else:
            return "Mixed Positive"
    else:
        if rating>=4:
            return "Positive"
        elif rating<=2:
            return "Negative"
        else:
            return "Neutral"

In [43]:
categorize_sentiment(-0.3089,3)

'Mixed Negative'

In [40]:
customer_reviews_df["sentiment_bucket"]=customer_reviews_df['sentiment_score'].apply(sentiment_bucket)

In [41]:
customer_reviews_df['sentiment_bucket']=customer_reviews_df['sentiment_score'].apply(sentiment_bucket)


In [44]:
customer_reviews_df["sentiment_category"]=customer_reviews_df.apply(lambda row:categorize_sentiment(row['sentiment_score'],row["Rating"]),axis=1)

In [45]:
customer_reviews_df

,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText,sentiment_score,sentiment_bucket,sentiment_category
0,1,77,18,2023-12-23,3,"Average experience, nothing special.",-0.3089,-0.49 to -0.5,Mixed Negative
1,2,80,19,2024-12-25,5,The quality is top-notch.,0.0000,0.0 to 0.49,Positive
2,3,50,13,2025-01-26,4,Five stars for the quick delivery.,0.0000,0.0 to 0.49,Positive
3,4,78,15,2025-04-21,3,"Good quality, but could be cheaper.",0.2382,0.0 to 0.49,Mixed Positive
4,5,64,2,2023-07-16,3,"Average experience, nothing special.",-0.3089,-0.49 to -0.5,Mixed Negative
...,...,...,...,...,...,...,...,...,...
1358,1359,28,4,2023-05-25,3,Not worth the money.,-0.1695,-0.49 to -0.5,Mixed Negative
1359,1360,58,12,2023-11-13,2,"Average experience, nothing special.",-0.3089,-0.49 to -0.5,Negative
1360,1361,96,15,2023-03-07,5,Customer support was very helpful.,0.6997,0.5 to 1.0,Positive
1361,1362,99,2,2025-12-03,1,Product did not meet my expectations.,0.0000,0.0 to 0.49,Negative


In [51]:
customer_reviews_df.to_csv("Fact_customer_review_Sentiment.csv",index=False)

PermissionError: [Errno 13] Permission denied: 'Fact_customer_review_Sentiment.csv'

In [53]:
customer_reviews_df.to_csv("Fact_customer_review_Sentiment.csv",index=False)

PermissionError: [Errno 13] Permission denied: 'Fact_customer_review_Sentiment.csv'